# Further error analysis with SHAP and LIME

Source target 0 means gallstones present; the modeled positive class is 1. The fixed seed reproduces the existing 70/15/15 splitting convention, then trains on the 85% development partition. Preprocessing, RFE, and stacking internal CV learn only from development rows. These are fresh experimental models, not the saved full-data refits or the original LOPO fold models. Choose the model before inspecting test outcomes.

Kernel SHAP explains the complete pipeline on raw features in probability units; LIME supplies a local approximation. Both use development data only. Correlated body-composition fields and synthetic perturbations can make attributions unstable or implausible. Explanations describe model behavior, not causal effects or clinical advice.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "xai_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside Experimental or its notebooks folder")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import display
from xai_utils import setup, explainers, shap_values, metrics
MODEL_NAME = "stacking"  # logistic, random_forest, xgboost, lightgbm, stacking
SEED = 42
SHAP_SAMPLES = 256  # Increase for more stable approximations; stacking can be slow.
LIME_SAMPLES = 3000
model, train, test, y_test, predict_proba, review, OUT = setup(MODEL_NAME, SEED)
kernel, lime = explainers(train, predict_proba, SEED)
display(pd.DataFrame([metrics(y_test, review.probability)]))
print(f"Training: {len(train)}; untouched test: {len(test)}; outputs: {OUT}")

## Error review, discrimination and calibration

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.calibration import CalibrationDisplay
review.to_csv(OUT / "heldout_error_review.csv", index_label="source_row")
display(review.groupby("error_type").agg(rows=("actual", "size"), mean_probability=("probability", "mean")))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, review.prediction, ax=axes[0], display_labels=["absent", "present"])
RocCurveDisplay.from_predictions(y_test, review.probability, ax=axes[1])
CalibrationDisplay.from_predictions(y_test, review.probability, n_bins=5, strategy="quantile", ax=axes[2])
plt.tight_layout()
plt.savefig(OUT / "error_diagnostics.png", bbox_inches="tight")
plt.show()
plt.close("all")
# Descriptive sensitivity analysis only: never select a threshold on this test set.
thresholds = pd.DataFrame([metrics(y_test, review.probability, t) for t in [.2,.3,.4,.5,.6,.7,.8]])
thresholds.to_csv(OUT / "threshold_sensitivity.csv", index=False)
display(thresholds)

## Subgroup errors
Show denominators alongside rates. Small groups have high uncertainty. Comparisons are exploratory and do not establish fairness or causal mechanisms.

In [ ]:
group_rows = []
for feature in ["gender", "age"]:
    groups = review[feature] if feature == "gender" else pd.cut(review.age, bins=[0,40,60,np.inf], right=False)
    for label, rows in review.groupby(groups, observed=True, dropna=False):
        positives = int((rows.actual == 1).sum())
        negatives = int((rows.actual == 0).sum())
        fn = int((rows.error_type == "FN").sum())
        fp = int((rows.error_type == "FP").sum())
        group_rows.append(dict(feature=feature, group=str(label), rows=len(rows), positives=positives,
            negatives=negatives, FN=fn, FP=fp, FNR=fn/positives if positives else np.nan,
            FPR=fp/negatives if negatives else np.nan))
subgroups = pd.DataFrame(group_rows)
subgroups.to_csv(OUT / "subgroup_errors.csv", index=False)
display(subgroups)
display(review.groupby("error_type")[list(test.columns)].median().T)

## Explain confident errors and correct outcomes
Take up to two cases from each FN, FP, TP and TN category. If a category is absent, report that rather than inventing cases. Attributions concern each fresh held-out prediction.

In [ ]:
case_ids = []
for group in ["FN", "FP", "TP", "TN"]:
    rows = review[review.error_type == group].sort_values("confidence", ascending=False)
    print(group, "available:", len(rows))
    case_ids.extend(rows.head(2).index.tolist())
cases = test.loc[case_ids]
case_values = shap_values(kernel, cases, SHAP_SAMPLES, SEED)
base = float(kernel.expected_value)
attributions = pd.DataFrame(case_values, index=cases.index, columns=test.columns)
attributions.to_csv(OUT / "error_case_shap.csv", index_label="source_row")
display(review.loc[case_ids, ["actual", "probability", "prediction", "error_type", "confidence"]])
print("Maximum reconstruction residual:", np.abs(base + case_values.sum(axis=1) - predict_proba(cases)[:, 1]).max())
local_rows = []
for i, source_row in enumerate(case_ids):
    print("Source row:", source_row, "category:", review.loc[source_row, "error_type"])
    shap.plots.waterfall(shap.Explanation(values=case_values[i], base_values=base,
        data=cases.loc[source_row].to_numpy(), feature_names=list(test.columns)), max_display=10, show=False)
    plt.savefig(OUT / f"error_shap_{source_row}.png", bbox_inches="tight")
    plt.show()
    plt.close("all")
    explanation = lime.explain_instance(cases.loc[source_row].to_numpy(), predict_proba,
        labels=(1,), num_features=10, num_samples=LIME_SAMPLES)
    explanation.save_to_file(str(OUT / f"error_lime_{source_row}.html"))
    for condition, weight in explanation.as_list(label=1):
        local_rows.append(dict(source_row=source_row, error_type=review.loc[source_row,"error_type"],
            condition=condition, weight=weight, surrogate_r2=explanation.score,
            surrogate_prediction=float(np.asarray(explanation.local_pred).ravel()[0])))
lime_table = pd.DataFrame(local_rows)
lime_table.to_csv(OUT / "error_case_lime.csv", index=False)
display(lime_table)
# Direction and magnitude by sampled error group, retaining group counts.
grouped = attributions.assign(error_type=review.loc[case_ids,"error_type"])
display(grouped.groupby("error_type").mean().T)
display(grouped.groupby("error_type").size().rename("explained_cases"))

## LIME stability for the first reviewed case

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
source_row = case_ids[0]
weights, quality = [], []
for repeat_seed in [7, 21, 42]:
    _, repeated_lime = explainers(train, predict_proba, repeat_seed)
    explanation = repeated_lime.explain_instance(test.loc[source_row].to_numpy(), predict_proba,
        labels=(1,), num_features=10, num_samples=LIME_SAMPLES)
    weights.append(pd.Series(dict(explanation.as_list(label=1)), name=repeat_seed))
    quality.append({"seed":repeat_seed, "surrogate_r2":explanation.score})
stability = pd.concat(weights, axis=1).fillna(0)
stability["std_weight"] = stability.std(axis=1)
stability.to_csv(OUT / "lime_stability.csv")
display(stability)
display(pd.DataFrame(quality))

## Review conclusions
Record whether missed positives share contributions, whether confident false positives are driven by correlated fields, and whether local surrogate fit or seed instability weakens the interpretation. Compare errors with the sampled correct outcomes; group summaries use at most two cases per category. Any proposed threshold or model change needs validation on development data and a new untouched evaluation.

References: [Kernel SHAP](https://shap.readthedocs.io/en/latest/generated/shap.KernelExplainer.html), [LIME](https://lime-ml.readthedocs.io/en/latest/lime.html).